# Sentiment Analysis using Machine Learning

**Project type:** Data Analytics / Natural Language Processing (NLP)  
**Goal:** classify English reviews as positive, negative, or neutral.

This notebook loads `dataset.csv`, explores the labels, cleans text, converts text to TF-IDF features, trains a Logistic Regression classifier, evaluates it, and saves the model and vectorizer. The included dataset has only 30 rows, so metrics are illustrative and should not be treated as evidence of production performance. Add a larger, representative dataset before drawing reliable conclusions.

In [ ]:
import re
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42
DATA_PATH = "dataset.csv"


## 1. Load and inspect the dataset

In [ ]:
data = pd.read_csv(DATA_PATH)
print("Shape:", data.shape)
display(data.head())
print("\\nColumns:", data.columns.tolist())
print("\\nMissing values:\\n", data.isna().sum())
print("\\nSentiment counts:\\n", data["sentiment"].value_counts())


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x="sentiment", order=data["sentiment"].value_counts().index)
plt.title("Number of reviews by sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Review count")
plt.tight_layout()
plt.show()


## 2. Clean text
Lowercase text, remove URLs and punctuation/numbers, then remove common English stopwords. The same cleaning function is used by the Streamlit app to keep training and prediction preprocessing aligned.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    return " ".join(word for word in text.split() if word not in ENGLISH_STOP_WORDS)

data = data.dropna(subset=["review", "sentiment"]).copy()
data["sentiment"] = data["sentiment"].astype(str).str.lower().str.strip()
data["clean_review"] = data["review"].astype(str).apply(clean_text)
data = data[data["clean_review"].str.strip().ne("")]
display(data[["review", "clean_review", "sentiment"]].head(10))


## 3. Split data and transform text with TF-IDF

In [ ]:
X = data["clean_review"]
y = data["sentiment"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print(f"Total rows: {len(data)} | Train rows: {len(X_train)} | Test rows: {len(X_test)}")
print("TF-IDF training matrix:", X_train_tfidf.shape)


## 4. Train Logistic Regression and evaluate

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.2%}")
print("\\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
labels = list(model.classes_)
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.tight_layout()
plt.show()


## 5. Predict a new review

In [ ]:
def predict_sentiment(review):
    cleaned = clean_text(review)
    if not cleaned:
        raise ValueError("Enter a review with usable English words.")
    vector = vectorizer.transform([cleaned])
    prediction = model.predict(vector)[0]
    probabilities = model.predict_proba(vector)[0]
    confidence = float(max(probabilities) * 100)
    return {"review": review, "sentiment": prediction, "estimated_probability_percent": round(confidence, 2)}

predict_sentiment("I really loved this product and the service was excellent.")


## 6. Save model artifacts
Run this notebook from the project folder. These files are loaded by `app.py`.

In [ ]:
joblib.dump(model, "model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")
print("Saved model.pkl and vectorizer.pkl")


## Limitations and next steps
- The supplied dataset contains only 30 examples (10 per class); the 20% test split has just 6 examples. A single test result is highly unstable.
- Short, simple examples may not represent real-world language, sarcasm, spelling variation, or domain-specific reviews.
- Increase the dataset substantially, verify labels, use a stratified validation strategy or cross-validation, and compare suitable baselines before claiming reliable performance.
- The predicted probability is not a guarantee of correctness.